In [8]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import xml.etree.ElementTree as ET
import re
import math
import networkx as nx
import os

app = Flask(__name__)
CORS(app) # Para evitar bloques de seguridad


ruta_svg = r"C:\Users\eare2\Documents\Intelijj\SmartParking\src\main\webapp\assets\img\Mapa_SmartParking.svg"

#Mencion de uso de IA en extraccion de codigo XML en el .svg
def grafo_total(ruta_svg): #Ponemos como parametro el .svg que es la imagen ya vectorizada
    arbol = ET.parse(ruta_svg) #Parseamos (traducir) a una variable arbol todo el contenido del .svg como si fuese un arbol
    raiz = arbol.getroot() #Obtenemos la raiz del nodo del arbol que acabamos de inicializar
    ns = {'svg': 'http://www.w3.org/2000/svg'} #namespace para que la funcion elementTree busque etiquetas tipo
    # <path> o <line> o <g>
    
    G = nx.Graph() #Instanaciamos un objeto de tipo grafo donde se guardaran todos los nodos que se vayan encontrando
    
    grafo = None #Variable temporal
    
    # Vamos a recorrer de forma recursiva los elementos del svg buscando el path que declaré como grafo rutas
    # desde figma que es la ruta gris de la imagen de arriba
    for p in raiz.iter():
        if 'graforutas' in p.get('id', '').lower():
            grafo = p
            break
        
    if grafo == None:
        return ValueError("No se encontró el path graforutas")
    
    rutas = [] #Inicializamos una lista que almacena el valor del atributo osea las coordenadas de cada elemento en el path
    
    #Basicamente si el tag que contiene grafo actualmente siempre termina en </path> va a ir metiendo el valor de cada vector
    # en la lista rutas, se le pone d, porque esta es la variable en el .svg que contiene toda la ruta xml
    if grafo.tag.endswith('path'):
        rutas.append(grafo.get('d', ''))
    else:
        for path in grafo.findall('.//svg:path', ns):
            rutas.append(path.get('d', '')) # Si no es </path> busca los elementos descendientes de path utilizando
            #el namespace y agrega los atributos d de cada uno en la lista
    #Necesitamos extraer los comandos en las cadenas de texto puesto que estas nos indicaran el orden de trazado
    for atributo in rutas:
        #Extraemos letras tanto mayusculas como minusculas, numeros negativos y positivos, decimales y enteros
        # , con la funcion re, separa cadena de texto del trazado en elementos individuales y se guarda en la variable tokens
        tokens = re.findall(r'[A-Za-z]|[-+]?\d*\.?\d+', atributo)
        
        actual = 'M' #M representa Move to, indicamos el estado inicial del trazador
        i = 0
        x_actual, y_actual = 0.0, 0.0
        nodo_anterior = None #Necesitamos guardar el padre del nodo siguiente para ir conectando los nodos
    
        while i < len(tokens):
            token = tokens[i] #Comenzamos el bucle para procesar el token uno por uno
            
            #Vienen las condicionales
            
            #Si el token es una letra
            if token.isalpha():
                actual = token.upper() #Mayusculas para estandarizar
                i += 1 #Incrementamos el indice para leer el siguiente token
                continue
            
            if actual in ['M', 'L']: # L = Line to, es para lineas rectas que no son ni verticales ni horizontales
                #osea estan como en diagonal pero son rectas, M es para moverse de punto en punto
                # Ambos comandos tanto M L les extramos las coordenadas en x,y mediante indices
                if i + 1 < len(tokens):
                    x_actual = float(tokens[i])
                    y_actual = float(tokens[i+1])
                    i += 2
            
            elif actual == 'H': #Para lineas horizontales
                x_actual = float(tokens[i])
                i += 1 
                
            elif actual == 'V': #Para lineas verticales
                y_actual = float(tokens[i])
                i += 1
            
            elif actual == 'C': #Para curvas
                #En estos casos figma siempre da 6 coordenadas para cuando son curvas
                if i + 5 < len(tokens):
                    # Agregamos los puntos intermedios de la curva para que no se vea como recta
                    cx1, cy1 = float(tokens[i]), float(tokens[i+1])
                    cx2, cy2 = float(tokens[i+2]), float(tokens[i+3])
                    x_actual, y_actual = float(tokens[i+4]), float(tokens[i+5])
                    
                    G.add_node((cx1, cy1), pos=(cx1, cy1))
                    G.add_node((cx2, cy2), pos=(cx2, cy2))
                    G.add_node((x_actual, y_actual), pos=(x_actual, y_actual))
                    
                    if nodo_anterior: #En la prinera iteracion no entra porque está vacio el nodo_anterior
                        G.add_edge(nodo_anterior, (cx1, cy1), weight=math.hypot(cx1 - nodo_anterior[0], cy1 - nodo_anterior[1]))
                    G.add_edge((cx1, cy1), (cx2, cy2), weight=math.hypot(cx2 - cx1, cy2 - cy1))
                    G.add_edge((cx2, cy2), (x_actual, y_actual), weight=math.hypot(x_actual - cx2, y_actual - cy2))
                    
                    nodo_anterior = (x_actual, y_actual) #Aqui vamos guardando el historial de cada nodo
                    i += 6
                    continue
            else: #Para otros comandos no soportados
                i += 1
                continue
            #Agregamos al nodo actual para las rectas
            nodo_actual = (x_actual, y_actual)
            G.add_node(nodo_actual, pos=nodo_actual)
            
            # Conectamos con el nodo anterior si no levantamos la pluma (M), me refiero a levantar la pluma cuando
            # en figma presiono esc para terminar el punto del vector 
            if actual != 'M' and nodo_anterior and nodo_anterior != nodo_actual:
                distancia = math.hypot(x_actual - nodo_anterior[0], y_actual - nodo_anterior[1])
                G.add_edge(nodo_anterior, nodo_actual, weight=distancia)  #Conectamos el nodo_padre con el nodo_actual
                #con la funcion add_edge que conecta mediante una arista los dos nodos y se le agrega una distancia calculada
                #para darle peso para despues indicarle al algoritmo el costo
                
            nodo_anterior = nodo_actual
            
            
    return G
        
grafo_parking = grafo_total(ruta_svg)
print("Numero de nodos conectados: ",grafo_parking.number_of_nodes())   

#Implementación del algoritmo de A*, sin métricas por el momento, solo tomando en cuenta coord inicial y final
def heuristica(nodo_a, nodo_b):
    return math.hypot(nodo_b[0] - nodo_a[0], nodo_b[1] - nodo_a[1])
#Calculamos la distancia euclidiana con la formula del teorema de pitagoras, para distancia entre dos puntos
# A diferencia de la distancia manhattan, que manhattan se especializa cuando son movimientos reestringidos 
# generalmente movimientos en 4 direcciones para simulaciones de robots, pacman, etc, no permitiendo diagonales

#Esta funcion es la mas costosa del algoritmo, porque es fuerza bruta, esta funcion parte del nodo inicial y final 
# para poder calcular y comparar de entre todos los nodos su hipotenusa para ver cual es la menor, y como bien sabemos
# entre menor la hipotenusa menor la distancia, mas cercano el nodo de acuerdo al nodo de donde partio a explorar 
def buscar_nodo_cercano(grafo, x_aprox, y_aprox):
    return min(grafo.nodes(), key=lambda n: math.hypot(n[0] - x_aprox, n[1] - y_aprox))


cajones_info = {}

def cargar_cajones_desde_svg(ruta_svg):
    arbol = ET.parse(ruta_svg)
    raiz = arbol.getroot()
    
    # Iteramos sobre TODOS los elementos del SVG
    for elemento in raiz.iter():
        id_elemento = elemento.get('id', '')
        
        # Si el elemento tiene guion
        if "-" in id_elemento:
            
            x_cajon, y_cajon = None, None
            
            def extraer_coordenadas(nodo):
                # Si es rectángulo
                if nodo.tag.endswith('rect'):
                    try:
                        x = float(nodo.get('x', 0)) + float(nodo.get('width', 0))/2
                        y = float(nodo.get('y', 0)) + float(nodo.get('height', 0))/2
                        return x, y
                    except: return None, None
                # Si es vector / path
                elif nodo.tag.endswith('path'):
                    d = nodo.get('d', '')
                    coords = re.findall(r'[-+]?\d*\.?\d+', d)
                    if len(coords) >= 2:
                        return float(coords[0]), float(coords[1])
                return None, None

            # Intentamos sacar la coordenada del elemento directamente
            x_cajon, y_cajon = extraer_coordenadas(elemento)
            
            # Si no funcionó, buscamos en lo que tenga adentro debido a que figma crea viñetas <g>
            if x_cajon is None:
                for hijo in elemento.iter():
                    x_cajon, y_cajon = extraer_coordenadas(hijo)
                    if x_cajon is not None:
                        break
            
            # Si logramos extraer las coordenadas
            if x_cajon is not None and y_cajon is not None:
                sufijo = id_elemento.split('-')[-1].upper()
                
                # Limpiamos el sufijo por si Figma agrego caracteres extra
                sufijo_limpio = ''.join(filter(str.isalpha, sufijo))
                
                tipo = "COCHE"
                es_azul = False
                es_vip = False
                
                if sufijo_limpio == 'C':
                    tipo = "COCHE"
                elif sufijo_limpio == 'T':
                    tipo = "CAMIONETA"
                elif sufijo_limpio == 'M':
                    tipo = "MOTO"
                elif sufijo_limpio == 'D':
                    tipo = "COCHE" 
                    es_azul = True
                elif sufijo_limpio == 'V':
                    tipo = "VIP"
                    es_vip = True
                else:
                    continue # No es un cajón válido
                    
                cajones_info[id_elemento] = {
                    "x": x_cajon,
                    "y": y_cajon,
                    "tipo": tipo,
                    "es_azul": es_azul,
                    "es_vip": es_vip
                }
                
    print(f"Cajones detectados listos para usarse: {len(cajones_info)}")

cargar_cajones_desde_svg(ruta_svg)

@app.route('/calcular_ruta', methods=['GET'])
def calcular_ruta():
    try:
        # Coordenadas
        origen_x = float(request.args.get('origen_x'))
        origen_y = float(request.args.get('origen_y'))
        destino_x = float(request.args.get('destino_x'))
        destino_y = float(request.args.get('destino_y'))
        
        # MÉTRICAS CON STRIP() PARA EVITAR ESPACIOS INVISIBLES
        es_discapacitado = request.args.get('es_discapacitado', 'false').strip().lower() == 'true'
        tipo_vehiculo = request.args.get('tipo_vehiculo', 'COCHE').strip().upper()
        es_vip = request.args.get('es_vip', 'false').strip().lower() == 'true'
        cajon_deseado = request.args.get('cajon_deseado', '').strip()

        print(f"Vehículo: {tipo_vehiculo} | VIP: {es_vip} | Discapacitado: {es_discapacitado}")

        cajon_optimo_id = None
        cajon_optimo = None
        
        if es_vip and cajon_deseado:
            print(f"Modo VIP {cajon_deseado}")
            cajon_optimo_id = cajon_deseado
            cajon_optimo = {"x": destino_x, "y": destino_y}
        else:
            print("Modo ESTÁNDAR")
            cajones_validos = {}
            
            for cid, datos in cajones_info.items():
                
                # Si el usuario ES discapacitado
                if es_discapacitado:
                    if datos["es_azul"]: # Solo agregamos si el cajón es azul
                        cajones_validos[cid] = datos
                        
                # Si el usuario es NORMAL
                else:
                    # El cajón NO debe ser azul, NO debe ser VIP, y el TIPO debe ser exacto al de su coche
                    if not datos["es_azul"] and not datos["es_vip"] and datos["tipo"] == tipo_vehiculo:
                        cajones_validos[cid] = datos


            # Si el filtro fue tan estricto que no quedó ninguno
            if len(cajones_validos) == 0:
                print("Error: No se encontraron cajones para este tipo de vehículo.")
                return jsonify({"status": "error", "mensaje": f"No hay espacios disponibles para {tipo_vehiculo}"}), 404

            # De los que pasaron, buscar el más cercano a la tienda
            distancia_minima = float('inf')
            for cid, datos in cajones_validos.items():
                dist = math.hypot(datos["x"] - destino_x, datos["y"] - destino_y)
                if dist < distancia_minima:
                    distancia_minima = dist
                    cajon_optimo_id = cid
                    
            cajon_optimo = cajones_validos[cajon_optimo_id]
            print(f"Cajón {cajon_optimo_id}")

        # Trazamos la ruta con A*
        inicio = buscar_nodo_cercano(grafo_parking, origen_x, origen_y)
        destino_final = buscar_nodo_cercano(grafo_parking, cajon_optimo["x"], cajon_optimo["y"])
        
        ruta_optima = nx.astar_path(grafo_parking, inicio, destino_final, heuristic=heuristica, weight='weight')
        print("Ruta trazada exitosamente.")
        
        return jsonify({
            "status": "success",
            "cajon_asignado": cajon_optimo_id, 
            "ruta": ruta_optima 
        }), 200
        
    except nx.NetworkXNoPath:
        return jsonify({"status": "error", "mensaje": "No se encontró un camino válido."}), 400
    except Exception as e:
        return jsonify({"status": "error", "mensaje": str(e)}), 500

if __name__ == '__main__':
    app.run(debug=True, use_reloader=False, port=5000)

Numero de nodos conectados:  677
Cajones detectados listos para usarse: 74
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [07/Jun/2026 02:37:18] "GET /calcular_ruta?origen_x=80.5&origen_y=262.5&destino_x=615&destino_y=9&es_discapacitado=false&tipo_vehiculo=COCHE&es_vip=false&cajon_deseado= HTTP/1.1" 200 -


Vehículo: COCHE | VIP: False | Discapacitado: False
Modo ESTÁNDAR
Cajón B4-C
Ruta trazada exitosamente.


127.0.0.1 - - [07/Jun/2026 02:37:35] "GET /calcular_ruta?origen_x=80.5&origen_y=262.5&destino_x=140&destino_y=9&es_discapacitado=false&tipo_vehiculo=COCHE&es_vip=false&cajon_deseado= HTTP/1.1" 200 -


Vehículo: COCHE | VIP: False | Discapacitado: False
Modo ESTÁNDAR
Cajón A1-C
Ruta trazada exitosamente.


...

Vehículo: COCHE | VIP: False | Discapacitado: False
Modo ESTÁNDAR
Cajón B8-C
Ruta trazada exitosamente.


127.0.0.1 - - [07/Jun/2026 03:06:28] "GET /calcular_ruta?origen_x=80.5&origen_y=262.5&destino_x=675.427490234375&destino_y=38.19460105895996&es_discapacitado=false&tipo_vehiculo=COCHE&es_vip=true&cajon_deseado=B4-C HTTP/1.1" 200 -


Vehículo: COCHE | VIP: True | Discapacitado: False
Modo VIP B4-C
Ruta trazada exitosamente.


127.0.0.1 - - [07/Jun/2026 03:06:39] "GET /calcular_ruta?origen_x=80.5&origen_y=262.5&destino_x=688.572021484375&destino_y=382.6300048828125&es_discapacitado=false&tipo_vehiculo=COCHE&es_vip=true&cajon_deseado=C14-T HTTP/1.1" 200 -


Vehículo: COCHE | VIP: True | Discapacitado: False
Modo VIP C14-T
Ruta trazada exitosamente.
